# FreshLens: Combined Training and Evaluation

This is the clean source notebook for a new Colab training run. The code follows the executed experiment; session-recovery cells and the hard-coded old output path have been removed. The optional image-upload function is defined but not automatically called.

Configure Drive paths before running. GPU training is recommended. Source archives, audit outputs and the review ZIP must exist on Drive. This notebook does not include datasets or trained weights.

Outputs are intentionally cleared: this edited source has not been rerun. The unchanged `archive/Executed_Training_Record.ipynb` contains the historical experiment outputs, including the CPU recovery workflow. Use that record to review the reported results; do not use Run All on it for a new experiment.

The original numbered section titles are retained so they match the project discussion. Python cell positions may differ after removal of recovery cells.


# ITI Retail — Combined Data Training
This simplified notebook keeps the existing **84 general product categories**. It combines all usable sources, trains a simple CNN and MobileNetV2, evaluates them, and exports an application-ready model.

**Setup:** Upload `Retail_Combined_Training_Pack.zip` to `MyDrive/ITI_Retail_Project`. Open this notebook in a fresh Colab GPU runtime. Keep your original five dataset ZIPs, audit, and data-review output on Drive. Run cells in order.

**Flow:** review decisions → image preparation → exact duplicates → group splits → augmentation → CNN → transfer learning → fine-tuning → validation selection → test reports → prediction → export.

- All 756 multi previews were visually inspected. The decision file records included regions and exclusions. Filenames provide context, not automatic labels. These annotations have not received independent expert verification.
- All 49,236 3-body files occur byte-for-byte in 100x100; they are accounted for without repeating their images.
- Both 100x100 and original-size are processed. Matching filenames alone never cause deletion. Meta is a reference source, not extra images.
- Entire normalized variety folders remain together across the controlled sources. One-group categories remain in training because an independent controlled holdout cannot be constructed from the available grouping information.
- Multi capture years are kept together: 2018 validation, 2022 test, other known years training. Unknown dates are excluded. All crops from one photo remain together.
- Folder and year groups reduce related-frame leakage; they do not prove complete physical-object independence. The natural test covers only a subset of classes and is not a guarantee for arbitrary web photos.
- This is a new split protocol. Do not compare its accuracy directly with the previous image-level 98.99%.

Preparation reads several GB. A completed image cache and model checkpoints are saved to Drive. No old retail model is used to initialize this experiment.

**Reading the code:** Run the numbered code cells in order. The code uses named functions and ordinary loops instead of lambda expressions, comprehensions, generator expressions, or dictionary unpacking. Image preparation is visible in this notebook. Some pandas, NumPy, and TensorFlow operations are necessary for efficient image training; their purpose is explained beside the relevant step.

**Same experiment:** Labels, crop coordinates, data preparation, split rules, random seed, models, epoch limits, callbacks, selection rule, reports, and export names are preserved. Training again can still produce slightly different numbers because of random augmentation and hardware behavior. A compatible completed cache from the previous version can be reused.


## 1. Imports and settings
Use a GPU. If memory is insufficient, reduce BATCH_SIZE to 32. TensorFlow and Keras versions are recorded for application compatibility.

**Code cell 1.** Connect Drive, set paths, and read the class names.


In [ ]:
from pathlib import Path
from io import BytesIO
from datetime import datetime
import json
import zipfile
import shutil
import hashlib
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from google.colab import drive, files

**Code cell 2.** Continue step 1: connect Drive, set paths, and read the class names.


In [ ]:
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/ITI_Retail_Project")
AUDIT = PROJECT / "dataset_audit" / "20260906_144352"
REVIEW_ZIP = PROJECT / "data_review" / "20260906_150238_Data_Review.zip"
PACK_ZIP = PROJECT / "Retail_Combined_Training_Pack.zip"
WORK = Path("/content/retail_combined_work")
PACK = WORK / "pack"
DATA = WORK / "prepared"
OUTPUT = PROJECT / "combined_categories" / datetime.now().strftime("%Y%m%d_%H%M%S")
IMAGE_SIZE = 160
BATCH_SIZE = 64
SEED = 42
CNN_EPOCHS, HEAD_EPOCHS, FINE_EPOCHS = 12, 8, 10
keras.utils.set_random_seed(SEED)
assert PACK_ZIP.is_file(), "Upload the training pack ZIP to ITI_Retail_Project."
assert REVIEW_ZIP.is_file(), "Check REVIEW_ZIP path."
assert (AUDIT / "image_inventory.csv").is_file(), "Check AUDIT path."
for folder in [PACK, DATA, OUTPUT / "models", OUTPUT / "reports", OUTPUT / "references"]:
    folder.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACK_ZIP) as archive:
    for name in ["reviewed_regions.csv", "review_decisions.csv", "class_names.json", "category_mapping.json", "inference.py"]:
        (PACK / name).write_bytes(archive.read(name))

**Code cell 3.** Continue step 1: connect Drive, set paths, and read the class names.


In [ ]:
class_names = json.loads((PACK / "class_names.json").read_text())
class_to_index = {}
for index, name in enumerate(class_names):
    class_to_index[name] = index
inventory = pd.read_csv(AUDIT / "image_inventory.csv").fillna("")
print("TensorFlow:", tf.__version__, "Keras:", keras.__version__)
print("Classes:", len(class_names), "Output:", OUTPUT)

**Code cell 4. Shared image preparation.** These are ordinary Python functions. They resize with white padding, create one input batch, and return the top three predictions. The application receives the same functions.


In [ ]:
def image_to_jpeg(image, image_size=160):
    image = ImageOps.pad(image.convert('RGB'), (image_size, image_size),
                         method=Image.Resampling.BILINEAR, color=(255, 255, 255))
    buffer = BytesIO()
    image.save(buffer, format='JPEG', quality=95)
    return buffer.getvalue()


def prepare_image(content, image_size=160):
    with Image.open(BytesIO(content)) as original:
        image = ImageOps.exif_transpose(original).convert('RGB')
        content = image_to_jpeg(image, image_size)
    image = tf.io.decode_jpeg(content, channels=3)
    return tf.cast(image, tf.float32).numpy()[None, ...]


def predict_product(model, class_names, content, image_size=160):
    probabilities = model(prepare_image(content, image_size), training=False).numpy()[0]
    if len(probabilities) != len(class_names):
        raise ValueError('Class count does not match model output.')
    indices = np.argsort(probabilities)[::-1][:3]
    results = []
    for index in indices:
        result = {
            'product': class_names[index],
            'score': float(probabilities[index])
        }
        results.append(result)
    return results


## 2. Resolve labels and load reviewed regions
The region coordinates were recorded from EXIF-oriented previews. They will be applied to full-resolution originals. Capture-date filtering creates the final multi decision report.

**Code cell 5.** Read the reviewed crop table and keep each capture year in one split.


In [ ]:
with zipfile.ZipFile(REVIEW_ZIP) as archive:
    review = pd.read_csv(BytesIO(archive.read("multi_review_manifest.csv"))).fillna("")
    label_map = pd.read_csv(BytesIO(archive.read("resolved_label_mapping.csv"))).fillna("")
assert len(review) == 756 and review["image_id"].is_unique
regions = pd.read_csv(PACK / "reviewed_regions.csv")
annotations = regions.merge(review[["image_id", "original_member", "capture_time"]], on="image_id", validate="many_to_one")
assert len(annotations) == len(regions)
annotations["year"] = annotations["capture_time"].str[:4]
known_date = annotations["year"].str.fullmatch(r"20[0-9]{2}").fillna(False)
missing_date_ids = set(annotations.loc[~known_date, "image_id"])
annotations = annotations[known_date].copy()

**Code cell 6.** Continue step 2: read the reviewed crop table and keep each capture year in one split.


In [ ]:
annotations["split"] = annotations["year"].map({"2018": "validation", "2022": "test"}).fillna("train")
annotations["group"] = "multi_year_" + annotations["year"]
assert set(annotations["label"]) <= set(class_names)
assert annotations.groupby("image_id")["split"].nunique().max() == 1
assert ((annotations["left"] >= 0) & (annotations["right"] <= 1) & (annotations["left"] < annotations["right"])).all()
assert ((annotations["top"] >= 0) & (annotations["bottom"] <= 1) & (annotations["top"] < annotations["bottom"])).all()

**Code cell 7.** Continue step 2: read the reviewed crop table and keep each capture year in one split.


In [ ]:
decisions = pd.read_csv(PACK / "review_decisions.csv")
decisions.loc[decisions["image_id"].isin(missing_date_ids), ["decision", "reason"]] = ["Excluded", "Missing capture date; scene grouping is uncertain."]
decisions.loc[decisions["decision"] == "Candidate", "decision"] = "Included"
decisions = decisions.merge(review[["image_id", "filename_hint"]], on="image_id", validate="one_to_one")
decisions.to_csv(OUTPUT / "reports" / "multi_review_decisions.csv", index=False)
annotations.to_csv(OUTPUT / "reports" / "multi_annotations.csv", index=False)
display(annotations.groupby("split").agg(regions=("crop_id", "size"), photos=("image_id", "nunique"), categories=("label", "nunique")))

## 3. Account for all five sources and build the preparation list
Pear common 1 and Pear 1 share a conservative group, without claiming identical fruit. Source Training/Test labels are retained in the audit, but this experiment uses new group splits. All 3-body images have confirmed copies in 100x100.

**Code cell 8.** Account for all sources and list every image that will be prepared.


In [ ]:
source_files = {
    "100x100": "fruits-360-100x100-main.zip",
    "original-size": "fruits-360-original-size-main.zip",
    "3-body-problem": "fruits-360-3-body-problem-main.zip",
    "multi": "fruits-360-multi-main.zip",
    "meta": "fruits-360-meta-main.zip"
}
for name in source_files.values():
    assert (PROJECT / name).is_file(), "Missing archive: " + name
evidence = pd.read_csv(AUDIT / "exact_duplicates_across_sources.csv")
base_hashes = set(evidence.loc[evidence["source"] == "100x100", "sha256"])
three = evidence[evidence["source"] == "3-body-problem"]
assert len(three) == int((inventory["source"] == "3-body-problem").sum())
assert three["sha256"].isin(base_hashes).all()
for path in AUDIT.iterdir():
    if path.name.endswith("LICENSE") or path.name.endswith("README.md") or path.name in ["metadata_reference.json", "audit_summary.json", "source_summary.csv"]:
        shutil.copy2(path, OUTPUT / "references" / path.name)

**Code cell 9.** Continue step 3: account for all sources and list every image that will be prepared.


In [ ]:
def normalize_name(name):
    name = name.lower()
    name = name.replace("_", " ")
    name = " ".join(name.split())
    name = name.replace("apple red delicios", "apple red delicious")
    return name

lookup = {}
for row in label_map.to_dict("records"):
    source = row["source"]
    if source not in lookup:
        lookup[source] = {}
    lookup[source][row["original_label"]] = row["general_category"]

**Code cell 10.** Continue step 3: account for all sources and list every image that will be prepared.


In [ ]:
rows = []
controlled = inventory[inventory["source"].isin(["100x100", "original-size"])]
for index, row in enumerate(controlled.to_dict("records")):
    label = lookup[row["source"]][row["source_label"]]
    assert label in class_to_index
    variety = normalize_name(row["source_label"])
    if variety == "pear common 1":
        variety = "pear 1"
    rows.append({"sample_id": "controlled_" + str(index).zfill(7), "source": row["source"], "member": row["member"], "label": label,
                 "group": "controlled/" + variety, "domain": "controlled", "split": "", "left": 0, "top": 0, "right": 1, "bottom": 1})
for row in annotations.to_dict("records"):
    rows.append({"sample_id": row["crop_id"], "source": "multi", "member": row["original_member"], "label": row["label"],
                 "group": row["group"], "domain": "natural", "split": row["split"], "left": row["left"], "top": row["top"], "right": row["right"], "bottom": row["bottom"]})
plan = pd.DataFrame(rows)
assert plan["sample_id"].is_unique
plan.to_csv(OUTPUT / "reports" / "preparation_plan.csv", index=False)
print("Planned regions:", len(plan), "3-body repeated files accounted for:", len(three))
display(inventory.groupby("source").size().rename("image_files").to_frame())

## 4. Prepare images and cache them on Drive
Read original image bytes, correct EXIF orientation, crop reviewed regions, preserve aspect ratio with white padding to 160 × 160, and save JPEG quality 95. This preprocessing is shared with inference.py.

The first run processes several GB. A completed Drive cache survives runtime resets. Rerunning in the same runtime reuses completed local images. Hashes identify identical prepared JPEG bytes, not every near-duplicate.

**Code cell 11.** Prepare original images and save or restore the reusable cache.


In [ ]:
source_state = {}
for name in source_files.values():
    file_info = (PROJECT / name).stat()
    source_state[name] = [file_info.st_size, file_info.st_mtime_ns]
signature = plan.to_csv(index=False) + json.dumps(source_state, sort_keys=True) + "pad160-jpeg95-v1"
cache_key = hashlib.sha256(signature.encode()).hexdigest()[:16]
CACHE = PROJECT / "combined_categories" / ("prepared_" + cache_key + ".zip")
key_file = DATA / "cache_key.txt"
if key_file.is_file():
    assert key_file.read_text() == cache_key, "Different local preparation plan; start a fresh runtime."

**Code cell 12.** Continue step 4: prepare original images and save or restore the reusable cache.


In [ ]:
if CACHE.is_file() and not (DATA / "prepared_manifest.csv").is_file():
    print("Restoring Drive cache...")
    local_cache = WORK / "prepared_cache.zip"
    shutil.copy2(CACHE, local_cache)
    with zipfile.ZipFile(local_cache) as archive:
        archive.extractall(DATA)
    local_cache.unlink()

**Code cell 13.** Continue step 4: prepare original images and save or restore the reusable cache.


In [ ]:
if (DATA / "prepared_manifest.csv").is_file():
    assert key_file.read_text() == cache_key
    prepared = pd.read_csv(DATA / "prepared_manifest.csv").fillna("")
    errors = pd.read_csv(DATA / "read_errors.csv").fillna("")
else:
    key_file.write_text(cache_key)
    (DATA / "images").mkdir(exist_ok=True)
    prepared_rows = []
    error_rows = []
    for source in ["100x100", "original-size", "multi"]:
        local_zip = WORK / source_files[source]
        if not local_zip.is_file():
            print("Copying", source)
            shutil.copy2(PROJECT / source_files[source], local_zip)
        source_rows = plan[plan["source"] == source].to_dict("records")
        with zipfile.ZipFile(local_zip) as archive:
            for index, row in enumerate(source_rows, 1):
                filename = "images/" + row["sample_id"] + ".jpg"
                path = DATA / filename
                try:
                    if not path.is_file():
                        with Image.open(BytesIO(archive.read(row["member"]))) as original:
                            image = ImageOps.exif_transpose(original).convert("RGB")
                            width, height = image.size
                            left = round(row["left"] * width)
                            top = round(row["top"] * height)
                            right = round(row["right"] * width)
                            bottom = round(row["bottom"] * height)
                            cropped_image = image.crop((left, top, right, bottom))
                            prepared_bytes = image_to_jpeg(cropped_image, IMAGE_SIZE)
                            path.write_bytes(prepared_bytes)
                    row["file"] = filename
                    row["sha256"] = hashlib.sha256(path.read_bytes()).hexdigest()
                    prepared_rows.append(row)
                except Exception as error:
                    error_rows.append({"source": source, "member": row["member"], "error": str(error)})
                if index % 5000 == 0 or index == len(source_rows):
                    print(source, index, "/", len(source_rows))
        local_zip.unlink()
    prepared = pd.DataFrame(prepared_rows)
    errors = pd.DataFrame(error_rows, columns=["source", "member", "error"])
    prepared.to_csv(DATA / "prepared_manifest.csv", index=False)
    errors.to_csv(DATA / "read_errors.csv", index=False)
    print("Saving Drive cache...")
    temporary_cache = CACHE.with_suffix(".partial.zip")
    with zipfile.ZipFile(temporary_cache, "w", zipfile.ZIP_STORED) as archive:
        for path in DATA.rglob("*"):
            if path.is_file():
                archive.write(path, str(path.relative_to(DATA)))
    temporary_cache.replace(CACHE)
errors.to_csv(OUTPUT / "reports" / "read_errors.csv", index=False)
print("Prepared:", len(prepared), "Read errors:", len(errors))
assert len(errors) == 0, "Resolve the saved read errors before training."
assert set(prepared["label"]) == set(class_names)

## 5. Remove exact duplicates and join related groups
Conflicting labels for identical prepared bytes are excluded and reported. A small dictionary joins groups that share an identical image. Keep one copy of same-label duplicates. A joined group containing held-out natural photos stays held out in full.

**Code cell 14.** Report label conflicts, join groups with shared images, and keep one exact copy.


In [ ]:
counts = prepared.groupby("sha256")["label"].nunique()
conflicting_hashes = set(counts[counts > 1].index)
conflicts = prepared[prepared["sha256"].isin(conflicting_hashes)]
conflicts.to_csv(OUTPUT / "reports" / "conflicting_labels.csv", index=False)
clean = prepared[~prepared["sha256"].isin(conflicting_hashes)].copy()

**Code cell 15.** Continue step 5: report label conflicts, join groups with shared images, and keep one exact copy.


In [ ]:
group_links = {}
for group in clean["group"].unique():
    group_links[group] = group
# Follow links until we reach the final name of a joined group.
# For example, if B is linked to A, both groups will use the name A.
def group_root(group):
    while group_links[group] != group:
        group = group_links[group]
    return group
shared = clean[clean.duplicated("sha256", keep=False)]
for _, table in shared.groupby("sha256"):
    groups = table["group"].unique()
    for group in groups[1:]:
        group_links[group_root(group)] = group_root(groups[0])
clean["group"] = clean["group"].map(group_root)

**Code cell 16.** Continue step 5: report label conflicts, join groups with shared images, and keep one exact copy.


In [ ]:
fixed_splits = {}
for group, table in clean.groupby("group"):
    values = set(table["split"])
    for split in ["test", "validation", "train"]:
        if split in values:
            fixed_splits[group] = split
            break
clean["priority"] = clean["split"].map({"test": 0, "validation": 1, "train": 2, "": 3}).fillna(3)
clean = clean.sort_values(["priority", "sample_id"])
removed = clean[clean.duplicated("sha256", keep="first")]
removed.to_csv(OUTPUT / "reports" / "removed_exact_duplicates.csv", index=False)
clean = clean.drop_duplicates("sha256").drop(columns="priority").reset_index(drop=True)
assert set(clean["label"]) == set(class_names), "Review conflicting-label report."
print("Conflicting rows:", len(conflicts), "Exact duplicates removed:", len(removed), "Usable:", len(clean))

## 6. Freeze group splits and report class coverage
Controlled classes with 3+ groups reserve about 15% of groups for validation and 15% for test, at least one each. Two groups: one training, one test. One group: training only. Natural years retain their fixed split. Unrecorded object overlap may remain; these are grouping proxies.

Each test subset can cover fewer than 84 classes. Inspect the coverage report; zero test support does not mean a class was evaluated successfully.

**Code cell 17.** Assign whole groups to training, validation, or test.


In [ ]:
rng = np.random.default_rng(SEED)
group_splits = {}
groups_table = clean[clean["domain"] == "controlled"][["label", "group"]].drop_duplicates()
for label, table in groups_table.groupby("label", sort=True):
    groups = []
    for group in table["group"].tolist():
        if group not in fixed_splits:
            groups.append(group)
    groups.sort()
    rng.shuffle(groups)
    for group in groups:
        group_splits[group] = "train"
    if len(groups) == 2:
        group_splits[groups[0]] = "test"
    elif len(groups) >= 3:
        count = max(1, int(len(groups) * 0.15))
        for group in groups[:count]:
            group_splits[group] = "test"
        for group in groups[count:2 * count]:
            group_splits[group] = "validation"

**Code cell 18.** Continue step 6: assign whole groups to training, validation, or test.


In [ ]:
group_splits.update(fixed_splits)
clean["split"] = clean["group"].map(group_splits)
clean["target"] = clean["label"].map(class_to_index)
image_paths = []
for filename in clean["file"]:
    image_paths.append(str(DATA / filename))
clean["path"] = image_paths
assert clean.groupby("group")["split"].nunique().max() == 1
assert clean.groupby("sha256")["split"].nunique().max() == 1
assert set(clean.loc[clean["split"] == "train", "label"]) == set(class_names)
assert set(clean["split"]) == {"train", "validation", "test"}
for domain in ["controlled", "natural"]:
    assert ((clean["domain"] == domain) & (clean["split"] == "validation")).any(), "Insufficient validation groups for " + domain
coverage = pd.crosstab(clean["label"], [clean["domain"], clean["split"]]).reindex(class_names, fill_value=0)
coverage.to_csv(OUTPUT / "reports" / "class_coverage.csv")
clean.to_csv(OUTPUT / "reports" / "split_manifest.csv", index=False)
display(clean.groupby(["domain", "split"]).agg(images=("sample_id", "size"), categories=("label", "nunique"), groups=("group", "nunique")))
display(coverage)

## 7. Inspect training samples and class distribution
The classifier expects one dominant product type. A reviewed region may contain several products of that same type; mixed types require cropping.

**Code cell 19.** Display training images and category counts.


In [ ]:
train_table = clean[clean["split"] == "train"].copy()
validation_table = clean[clean["split"] == "validation"].copy()
test_table = clean[clean["split"] == "test"].copy()
natural_train = train_table[train_table["domain"] == "natural"]
assert len(natural_train) > 0
examples = natural_train.sample(n=min(12, len(natural_train)), random_state=SEED)
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for axis in axes.flat:
    axis.axis("off")
for axis, row in zip(axes.flat, examples.to_dict("records")):
    axis.imshow(Image.open(row["path"]))
    axis.set_title(row["label"])
plt.tight_layout()
plt.savefig(OUTPUT / "reports" / "training_examples.png", dpi=140)
plt.show()
train_table["label"].value_counts().sort_index().plot.bar(figsize=(18, 4), title="Training images per general category")
plt.tight_layout()
plt.savefig(OUTPUT / "reports" / "class_distribution.png", dpi=140)
plt.show()

## 8. Build datasets with more exposure to natural backgrounds
Every controlled and natural training region appears at least once per epoch. Natural regions are repeated to approximately 25% of training rows, unless already more numerous. Repeats are not new independent data.

Validation sample weights give controlled and natural domains equal total weight in val_loss. Accuracy remains unweighted. Test images are never repeated or augmented.

**Code cell 20.** Create the training mixture and TensorFlow image datasets.


In [ ]:
controlled_train = train_table[train_table["domain"] == "controlled"]
natural_count = max(len(natural_train), len(controlled_train) // 3)
extra = natural_train.sample(n=natural_count - len(natural_train), replace=True, random_state=SEED)
training_rows = pd.concat([controlled_train, natural_train, extra], ignore_index=True).sample(frac=1, random_state=SEED)

**Code cell 21.** Continue step 8: create the training mixture and TensorFlow image datasets.


In [ ]:
def read_prepared_image(path):
    return tf.cast(tf.io.decode_jpeg(tf.io.read_file(path), channels=3), tf.float32)
def read_image_and_label(path, label):
    image = read_prepared_image(path)
    return image, label

def read_image_label_and_weight(path, label, weight):
    image = read_prepared_image(path)
    return image, label, weight

def make_dataset(table, training=False, balanced=False):
    paths = table["path"].astype(str).to_numpy()
    targets = table["target"].to_numpy(dtype="int32")
    if balanced:
        sizes = table["domain"].value_counts()
        weights = []
        for domain in table["domain"]:
            weight = len(table) / (len(sizes) * sizes[domain])
            weights.append(weight)
        weights = np.array(weights, dtype="float32")
        dataset = tf.data.Dataset.from_tensor_slices((paths, targets, weights))
        if training:
            dataset = dataset.shuffle(len(table), seed=SEED)
        dataset = dataset.map(read_image_label_and_weight, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        dataset = tf.data.Dataset.from_tensor_slices((paths, targets))
        if training:
            dataset = dataset.shuffle(len(table), seed=SEED)
        dataset = dataset.map(read_image_and_label, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

**Code cell 22.** Continue step 8: create the training mixture and TensorFlow image datasets.


In [ ]:
train_ds = make_dataset(training_rows, training=True)
val_ds = make_dataset(validation_table, balanced=True)
print("Rows per epoch:", len(training_rows), "Natural fraction:", natural_count / len(training_rows))

## 9. Augmentation and training helpers
Augmentation runs only during training. Normalization stays inside the model. EarlyStopping may end before the maximum epoch count. To restart an experiment, rerun its model-construction cell before training.

**Code cell 23.** Define augmentation and standard Keras training settings.


In [ ]:
def make_augmentation():
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08, fill_mode="constant", fill_value=255),
        layers.RandomTranslation(0.08, 0.08, fill_mode="constant", fill_value=255),
        layers.RandomZoom(0.10, fill_mode="constant", fill_value=255),
        layers.RandomContrast(0.15),
        layers.RandomBrightness(0.12, value_range=(0, 255))
    ], name="augmentation")

**Code cell 24.** Continue step 9: define augmentation and standard Keras training settings.


In [ ]:
def compile_model(model, learning_rate):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
def make_callbacks(name):
    return [
        keras.callbacks.ModelCheckpoint(str(OUTPUT / "models" / (name + ".keras")), monitor="val_loss", save_best_only=True),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=2, factor=0.5, min_lr=1e-7),
        keras.callbacks.CSVLogger(str(OUTPUT / "reports" / (name + "_history.csv")))
    ]

## 10. Build a simple CNN from scratch

**Code cell 25.** Create the CNN with the same layers and settings.


In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)
cnn = keras.Sequential([
    keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)), make_augmentation(), layers.Rescaling(1.0 / 255),
    layers.Conv2D(32, 3, padding="same", activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(128, 3, padding="same", activation="relu"), layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(), layers.Dense(128, activation="relu"), layers.Dropout(0.35),
    layers.Dense(len(class_names), activation="softmax")
], name="retail_cnn")
compile_model(cnn, 0.001)
cnn.summary()

## 11. Train the CNN

**Code cell 26.** Train the CNN and save its best checkpoint.


In [ ]:
cnn_history = cnn.fit(train_ds, validation_data=val_ds, epochs=CNN_EPOCHS, callbacks=make_callbacks("cnn_best"))
del cnn
gc.collect()

## 12. Build MobileNetV2 transfer learning
Start with ImageNet weights. The base is frozen for head training. Calling the base with training=False keeps batch-normalization statistics fixed.

**Code cell 27.** Create MobileNetV2 with ImageNet weights.


In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)
base = keras.applications.MobileNetV2(input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False, weights="imagenet")
base.trainable = False
inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = make_augmentation()(inputs)
x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.30)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)
transfer = keras.Model(inputs, outputs, name="retail_mobilenet")
compile_model(transfer, 0.001)
transfer.summary()

## 13. Train the new classifier head

**Code cell 28.** Train the new classification head.


In [ ]:
head_history = transfer.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, callbacks=make_callbacks("transfer_head_best"))

## 14. Fine-tune the upper layers
Reload the best head checkpoint. Unfreeze the last 30 base layers except BatchNormalization, recompile, and use a smaller learning rate.

**Code cell 29.** Load the best head, unfreeze upper layers, and fine-tune.


In [ ]:
transfer = keras.models.load_model(OUTPUT / "models" / "transfer_head_best.keras", compile=False)
base = None
for layer in transfer.layers:
    if isinstance(layer, keras.Model):
        if "mobilenet" in layer.name.lower():
            base = layer
            break
assert base is not None, "MobileNetV2 base was not found."
base.trainable = True
for layer in base.layers:
    layer.trainable = False
for layer in base.layers[-30:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True

**Code cell 30.** Continue step 14: load the best head, unfreeze upper layers, and fine-tune.


In [ ]:
compile_model(transfer, 1e-5)
fine_history = transfer.fit(train_ds, validation_data=val_ds, epochs=FINE_EPOCHS, callbacks=make_callbacks("transfer_fine_best"))
del transfer, base
gc.collect()

## 15. Plot learning curves
Validation loss balances two domains, while training loss uses the training mixture. Their absolute values describe different distributions. Curves alone do not prove generalization.


```
# This is formatted as code
```

**Code cell 31.** Plot all training and validation curves.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path


model_names = [
    "cnn_best",
    "transfer_head_best",
    "transfer_fine_best"
]

fig, axes = plt.subplots(3, 2, figsize=(12, 12))

for row, name in enumerate(model_names):
    history = pd.read_csv(
        OUTPUT / "reports" / (name + "_history.csv")
    )

    epochs = range(1, len(history) + 1)

    axes[row, 0].plot(
        epochs, history["accuracy"], label="Training accuracy"
    )
    axes[row, 0].plot(
        epochs, history["val_accuracy"], label="Validation accuracy"
    )
    axes[row, 0].set_title(name + " - Accuracy")
    axes[row, 0].set_xlabel("Epoch")
    axes[row, 0].set_ylabel("Accuracy")
    axes[row, 0].legend()
    axes[row, 0].grid(True)

    axes[row, 1].plot(
        epochs, history["loss"], label="Training loss"
    )
    axes[row, 1].plot(
        epochs, history["val_loss"], label="Validation loss"
    )
    axes[row, 1].set_title(name + " - Loss")
    axes[row, 1].set_xlabel("Epoch")
    axes[row, 1].set_ylabel("Loss")
    axes[row, 1].legend()
    axes[row, 1].grid(True)

plt.tight_layout()
plt.savefig(
    OUTPUT / "reports" / "training_curves.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

## 16. Select using validation only
The fixed selection rule is the lowest mean of controlled and natural validation loss. Macro scores average classes actually present in that subset. Coverage is reported alongside every score.

**Code cell 32.** Compare validation results and lock the selected model.


In [ ]:
def score_predictions(table, probabilities):
    truth = table["target"].to_numpy()
    predicted = probabilities.argmax(axis=1)
    present = np.unique(truth)
    precision, recall, f1, _ = precision_recall_fscore_support(truth, predicted, labels=present, average="macro", zero_division=0)
    true_scores = probabilities[np.arange(len(truth)), truth]
    return {"images": len(truth), "categories": len(present), "accuracy": float(accuracy_score(truth, predicted)),
            "macro_precision": float(precision), "macro_recall": float(recall), "macro_f1": float(f1),
            "loss": float(-np.log(np.clip(true_scores, 1e-7, 1)).mean())}

**Code cell 33.** Continue step 16: compare validation results and lock the selected model.


In [ ]:
validation_results = []

for name in model_names:
    print("Evaluating:", name)

    model_path = OUTPUT / "models" / (name + ".keras")
    model = keras.models.load_model(model_path, compile=False)

    for domain in ["controlled", "natural"]:
        table = validation_table[
            validation_table["domain"] == domain
        ].copy()

        assert len(table) > 0, "No validation images for " + domain

        print("Validation domain:", domain)

        dataset = make_dataset(table)
        probabilities = model.predict(dataset, verbose=1)

        scores = score_predictions(table, probabilities)

        result = {
            "model": name,
            "domain": domain
        }

        result.update(scores)
        validation_results.append(result)

    del model
    keras.backend.clear_session()
    gc.collect()

validation_report = pd.DataFrame(validation_results)

selection = (
    validation_report.groupby("model")["loss"]
    .mean()
    .sort_values()
)

selected_name = str(selection.index[0])

validation_report.to_csv(
    OUTPUT / "reports" / "validation_comparison.csv",
    index=False
)

selection.rename("balanced_validation_loss").to_csv(
    OUTPUT / "reports" / "model_selection.csv"
)

(OUTPUT / "models" / "selected_model.txt").write_text(
    selected_name
)

display(validation_report)
print("Selected using validation only:", selected_name)

## 17. Evaluate the frozen test subsets
Controlled and natural-background results are separate. The selected model stays locked even if another model scores higher on test. The natural subset is a year-held-out benchmark from this dataset collection; newly photographed external products are still needed for a stronger deployment claim.

**Code cell 34.** Evaluate the frozen test subsets without changing model selection.


In [ ]:
test_results = []
selected_test_probabilities = None
for name in model_names:
    model = keras.models.load_model(OUTPUT / "models" / (name + ".keras"), compile=False)
    probabilities = model.predict(make_dataset(test_table), verbose=1)
    for domain in ["controlled", "natural"]:
        mask = test_table["domain"].to_numpy() == domain
        if mask.any():
            table = test_table.loc[mask]
            domain_probabilities = probabilities[mask]
            scores = score_predictions(table, domain_probabilities)
            result = {"model": name, "domain": domain}
            result.update(scores)
            test_results.append(result)
    if name == selected_name:
        selected_test_probabilities = probabilities
    del model
    keras.backend.clear_session()
test_report = pd.DataFrame(test_results)
test_report.to_csv(OUTPUT / "reports" / "test_comparison.csv", index=False)
display(test_report)
print("Selection remains:", selected_name)

## 18. Inspect mistakes and per-class metrics
Reports include all 84 classes, with zero support where no test images exist. Do not interpret zero-support classes as evaluated.

**Code cell 35.** Save per-class reports and display prediction mistakes.


In [ ]:
predicted = selected_test_probabilities.argmax(axis=1)
truth = test_table["target"].to_numpy()
for domain in ["controlled", "natural"]:
    mask = test_table["domain"].to_numpy() == domain
    if not mask.any():
        continue
    report = classification_report(truth[mask], predicted[mask], labels=np.arange(len(class_names)), target_names=class_names, output_dict=True, zero_division=0)
    pd.DataFrame(report).transpose().to_csv(OUTPUT / "reports" / (domain + "_classification_report.csv"))
    matrix = confusion_matrix(truth[mask], predicted[mask], labels=np.arange(len(class_names)))
    pd.DataFrame(matrix, index=class_names, columns=class_names).to_csv(OUTPUT / "reports" / (domain + "_confusion_matrix.csv"))

**Code cell 36.** Continue step 18: save per-class reports and display prediction mistakes.


In [ ]:
results = test_table[["sample_id", "domain", "label", "group"]].copy()
predicted_names = []
for index in predicted:
    predicted_names.append(class_names[index])
results["predicted"] = predicted_names
results["score"] = selected_test_probabilities.max(axis=1)
results.to_csv(OUTPUT / "reports" / "test_predictions.csv", index=False)

**Code cell 37.** Continue step 18: save per-class reports and display prediction mistakes.


In [ ]:
wrong = np.flatnonzero(predicted != truth).tolist()
natural_mistakes = []
controlled_mistakes = []
for index in wrong:
    if test_table.iloc[index]["domain"] == "natural":
        natural_mistakes.append(index)
    else:
        controlled_mistakes.append(index)
wrong = natural_mistakes + controlled_mistakes
fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for axis in axes.flat:
    axis.axis("off")
for axis, i in zip(axes.flat, wrong[:12]):
    axis.imshow(Image.open(test_table.iloc[i]["path"]))
    axis.set_title("True: " + class_names[truth[i]] + "\nPred: " + class_names[predicted[i]], fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT / "reports" / "prediction_errors.png", dpi=140)
plt.show()

## 19. Save application files
Input is now 160 × 160 with aspect-ratio padding and a JPEG preprocessing step. Use the included inference helper with the new application. Replacing only the model inside an old preprocessing pipeline is insufficient.

**Code cell 38.** Export the model, labels, configuration, and inference helper.


In [ ]:
MODEL_DIR = OUTPUT / "models"
shutil.copy2(MODEL_DIR / (selected_name + ".keras"), MODEL_DIR / "best_model.keras")
(MODEL_DIR / "class_names.json").write_text(json.dumps(class_names, indent=2))
shutil.copy2(PACK / "category_mapping.json", MODEL_DIR / "category_mapping.json")
config = {"selected_model": selected_name, "label_level": "general_product", "image_size": IMAGE_SIZE,
          "num_classes": len(class_names), "color_mode": "RGB", "pixel_range": [0, 255],
          "normalization": "included_in_model", "resize_method": "pil_bilinear_white_pad_then_jpeg95",
          "tensorflow_version": tf.__version__, "keras_version": keras.__version__,
          "selection_rule": "minimum mean controlled/natural validation loss",
          "split_protocol": "normalized variety groups; multi 2018 validation, 2022 test, other known years train",
          "limitation": "Grouping proxies do not prove independent objects; natural test covers a subset; unknown objects may receive high scores."}
(MODEL_DIR / "model_config.json").write_text(json.dumps(config, indent=2))
shutil.copy2(PACK / "inference.py", OUTPUT / "inference.py")
print("Model:", MODEL_DIR / "best_model.keras")

## 20. Verify preprocessing and saved-model consistency
A full-image multi sample verifies identical prepared pixels between training and the app helper. Saved and original checkpoint predictions must agree. This checks implementation consistency, not real-world accuracy.

**Code cell 39.** Check that training and application preprocessing produce the same pixels.


In [ ]:
best_model = keras.models.load_model(MODEL_DIR / "best_model.keras", compile=False)
assert best_model.input_shape[1:] == (IMAGE_SIZE, IMAGE_SIZE, 3)
assert best_model.output_shape[-1] == len(class_names)
full_rows = clean[(clean["source"] == "multi") & (clean["left"] == 0) & (clean["top"] == 0) & (clean["right"] == 1) & (clean["bottom"] == 1)]
row = full_rows.iloc[0]
with zipfile.ZipFile(PROJECT / source_files["multi"]) as archive:
    original_bytes = archive.read(row["member"])
app_input = prepare_image(original_bytes, IMAGE_SIZE)
prepared_input = read_prepared_image(row["path"]).numpy()[None, ...]
np.testing.assert_array_equal(app_input, prepared_input)
original_model = keras.models.load_model(MODEL_DIR / (selected_name + ".keras"), compile=False)
np.testing.assert_allclose(best_model(app_input, training=False).numpy(), original_model(app_input, training=False).numpy(), rtol=1e-5, atol=1e-6)
del original_model
print("PASS: shape, labels, preprocessing pixels, and saved-model predictions.")

## 21. Optional manual prediction
Run `upload_and_predict()` in a new cell when you want to test your own photo. This cell only defines the function so Run all is not blocked by an upload dialog. Use one dominant product or crop the image first. Softmax scores are not guaranteed correctness or an unknown-object detector.

**Code cell 40.** Define the optional manual photo test.


In [ ]:
def upload_and_predict():
    uploaded = files.upload()
    for filename, content in uploaded.items():
        display(Image.open(BytesIO(content)))
        print(filename)
        display(pd.DataFrame(predict_product(best_model, class_names, content, IMAGE_SIZE)))

## 22. Export
The ZIP contains the selected model, labels, configuration, inference helper, reports, annotations, and source references. Other checkpoints remain on Drive. Send this export and the two-domain test table before updating the application.

**Code cell 41.** Package and download the final export.


In [ ]:
EXPORT = OUTPUT / "Retail_Combined_Model_Export.zip"
with zipfile.ZipFile(EXPORT, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in ["best_model.keras", "class_names.json", "model_config.json", "category_mapping.json", "selected_model.txt"]:
        archive.write(MODEL_DIR / name, "models/" + name)
    archive.write(OUTPUT / "inference.py", "inference.py")
    for folder in ["reports", "references"]:
        for path in (OUTPUT / folder).rglob("*"):
            if path.is_file():
                archive.write(path, str(path.relative_to(OUTPUT)))
print("COMBINED TRAINING COMPLETE")
print("Selected model:", selected_name)
print("Export:", EXPORT)
display(test_report[test_report["model"] == selected_name])
files.download(str(EXPORT))

## Discussion notes
- More files do not necessarily provide more independent information.
- Variety groups help prevent neighboring capture frames inflating validation scores.
- Singleton groups cannot supply an independent controlled test for their category.
- Natural-region repetition increases training exposure; it does not create new observations and can still overfit.
- Test-domain separation reveals errors hidden by a large white-background subset.
- This is classification, not object detection, counting, or inventory tracking.
- Natural images cover a subset of the 84 categories. Improvement must be measured after training; no accuracy is promised.

[TensorFlow transfer learning](https://www.tensorflow.org/tutorials/images/transfer_learning) · [Keras image loading](https://keras.io/api/data_loading/image/) · [Keras augmentation](https://keras.io/api/layers/preprocessing_layers/image_augmentation/)